In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats 
# import joblib
import re
# import pacmap
import random
# from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
# from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
# from scipy.spatial.distance import pdist
# from sklearn.metrics import silhouette_score
# from scipy.spatial.distance import squareform
# from sklearn.decomposition import PCA
from scipy.signal import medfilt
from itertools import groupby
import matplotlib.patches as patches

SUNFLOWER=42
random.seed(SUNFLOWER)
np.random.seed(SUNFLOWER)
def shapator(data):
    for key in data.keys():
        print(f"{key} shape: {data[key].shape}")

In [ ]:
# the very initial data (aggregate over 10 rows = 100s) and dont make the number of rows divisible by 10 to not lose data with extra column saying what resolution each row is accounting for the last row that might not be the same resolution
# get the chunk id for each data point and make sure to preserve the order of data(meaning if there is a gap within a chunk you should be able to locate it)
# Get BCI and reduce it to same resolution and split other in burst suppression and continuous and discontinuous.
# split data into training and testing

# so final result for Ahmed should be:
# features nx1275 where n will have all the patients (train and test respectively)and each patient can have different number of rows(however it has to be sorted in chronological order so patient 1 second 1 ->second 100 then patient 2 second 1 to second 98632)
# activation values for each data point
# predictions for each data point with labeling (0-7) with what each value corresponds to and other shouldn't be one of them
# add cpc_bin for each patient


In [ ]:
#'----------------------
#' Script name: chunkify_ppnet.py 
#' 
#' Version: 0.0.1
#' 
#' Authors: Michael P. Brown
#' 
#' Date created: 2026-04-05
#' 
#' Copyright (c) 2026
#' Email: mpbrown@berkeley.edu
#'----------------------
#### E.g. ProtoPNet file syntax:  ICARE_IdNumber_YearMonthDay_HourMinuteSecond
# ... [Directory setup and offsets loading remain the same] ...
mgh_dir = "../ICARE_protopnet_results/MGH/"
ulb_dir = "../ICARE_protopnet_results/ULB/"
ynh_dir = "../ICARE_protopnet_results/YNH/"
bidmc_dir = "../ICARE_protopnet_results/BIDMC/"
bwh_dir = "../ICARE_protopnet_results/BWH/"
utw_dir = "../ICARE_protopnet_results/UTW/"

npz_directory = "../Vishnu_IIC_SparcNet_Analysis/physionet_files_mapped.csv"
df_offset_ca = pd.read_csv(npz_directory)
df_offset_ca['Extented Ids'] = df_offset_ca['ICARE_files'].apply(lambda r: r[:-4])

npz_directories = [mgh_dir, ulb_dir, ynh_dir, bidmc_dir, bwh_dir, utw_dir]
offsets = df_offset_ca[['time_from_rosc', 'Extented Ids']] ## lenght 5631,  dim 2
offsets = dict(zip(offsets["Extented Ids"].values,offsets["time_from_rosc"].values))

all_features, all_predictions, all_activations, all_time_chunks, all_patient_ids = [], [], [], [], []

N = 10
SEC_PER_CHUNK = 21600 # 6 hours

# Assuming 'offsets' dictionary is already created from your snippet

for npz_dir in npz_directories:
    if not os.path.exists(npz_dir): continue
        
    for file in os.listdir(npz_dir):
        if not file.endswith(".npz"): continue
        file_base = file.replace(".npz", "")
        
        if file_base not in offsets:
            continue
# features  
        try:
            with np.load(os.path.join(npz_dir, file)) as data:
                feat = data["extracted_features"] if "extracted_features" in data else data["proto_features"]
                pred = np.atleast_1d(data["predictions"])
                acts = data['activations']
                
                if len(feat.shape) == 1:
                    feat, acts = feat.reshape(1, -1), acts.reshape(1, -1)
                # make cleanly divisible by window size
                cutoff = (feat.shape[0] // N) * N
                if cutoff == 0: continue 

                # average acorss window size
                feat_c = feat[:cutoff].reshape(-1, N, feat.shape[1]).mean(axis=1)
                pred_c = stats.mode(pred[:cutoff].reshape(-1, N), axis=1, keepdims=False)[0]
                acts_c = acts[:cutoff].reshape(-1, N, acts.shape[1]).mean(axis=1)
                
                # Calculate true time chunks
                start_sec = offsets[file_base] + 20
                row_times = start_sec + (np.arange(0, cutoff, N) * 10) 
                chunk_ids = row_times // SEC_PER_CHUNK
                
                patient_id = "_".join(file.split('_')[:2])
                
                all_features.append(feat_c)
                all_predictions.append(pred_c)
                all_activations.append(acts_c)
                all_time_chunks.append(chunk_ids)
                all_patient_ids.extend([patient_id] * feat_c.shape[0])
                
        except Exception as e:
            print(f"Error loading {file}: {e}")

# Stack and filter 0-84 hours (chunks 0-13)
final_time_chunks = np.concatenate(all_time_chunks)
valid_mask = final_time_chunks < 14

final_features = np.vstack(all_features)[valid_mask]
final_predictions = np.concatenate(all_predictions)[valid_mask]
final_activations = np.vstack(all_activations)[valid_mask]
final_patient_ids = np.array(all_patient_ids)[valid_mask]
final_time_chunks = final_time_chunks[valid_mask]
# N=3 overloads memory
# N=10 runtime 2m

In [ ]:
{k: v for k, v in offsets.items() if k.startswith('ICARE_0647_')}

In [ ]:
#'----------------------
#' Script name: chunkify_ppnet.py 
#' 
#' Version: 0.0.2
#' 
#' Authors: Michael P. Brown
#' 
#' Date created: 2026-04-05
#' 
#' Copyright (c) 2026
#' Email: mpbrown@berkeley.edu
#'----------------------
#### E.g. ProtoPNet file syntax:  ICARE_IdNumber_YearMonthDay_HourMinuteSecond

# --- 1. Configuration & Mappings ---
N = 10
SEC_PER_CHUNK = 21600 # 6 hours

# Directories
mgh_dir = "../ICARE_protopnet_results/MGH/"
ulb_dir = "../ICARE_protopnet_results/ULB/"
ynh_dir = "../ICARE_protopnet_results/YNH/"
bidmc_dir = "../ICARE_protopnet_results/BIDMC/"
bwh_dir = "../ICARE_protopnet_results/BWH/"
utw_dir = "../ICARE_protopnet_results/UTW/"
npz_directories = [mgh_dir, ulb_dir, ynh_dir, bidmc_dir, bwh_dir, utw_dir]

bci_csv_dir = "../1000_ICARE_patient_10s_94f_with_spike/" 

# Load Offsets
df_offset = pd.read_csv("../Vishnu_IIC_SparcNet_Analysis/physionet_files_mapped.csv")
df_offset['Extended_Ids'] = df_offset['ICARE_files'].apply(lambda r: str(r)[:-4])
offsets = dict(zip(df_offset["Extended_Ids"].values, df_offset["time_from_rosc"].values))

# Load CPC Scores
df_cpc = pd.read_csv('pats_cpc_outcomes.csv')
cpc_dict = dict(zip(df_cpc['pat_ICARE'], df_cpc['cpc']))

# Initialize Global Lists
all_features, all_predictions, all_activations = [], [], []
all_patient_ids, all_time_chunks, all_resolutions, all_cpc, all_times = [], [], [], [], []

# --- 2. Main Extraction Loop ---
for npz_dir in npz_directories:
    if not os.path.exists(npz_dir): continue
        
    for file in os.listdir(npz_dir):
        if not file.endswith(".npz"): continue
        file_base = file.replace(".npz", "")
        
        if file_base not in offsets: continue
            
        patient_id = "_".join(file_base.split('_')[:2]) # e.g., 'ICARE_0647'
        
        # Load Patient's BCI CSV
        bci_csv_path = os.path.join(bci_csv_dir, f"{patient_id}_rel10s_with_spike.csv")
        if not os.path.exists(bci_csv_path): continue
            
        try:
            # --- Load PPNet Data ---
            with np.load(os.path.join(npz_dir, file)) as data:
                feat = data["extracted_features"] if "extracted_features" in data else data["proto_features"]
                pred = np.atleast_1d(data["predictions"])
                acts = data['activations']
                
                if len(feat.shape) == 1:
                    feat, acts = feat.reshape(1, -1), acts.reshape(1, -1)
            
            # --- Load and Align BCI Data ---
            df_bci = pd.read_csv(bci_csv_path)
            # Filter rows specifically for this .mat recording
            df_file_bci = df_bci[df_bci['file'].str.contains(file_base, na=False)].sort_values('rel_sec')
            if df_file_bci.empty: continue
                
            # Align PPNet +20s skip with BCI (dropping first 2 rows = 20s)
            bci_aligned = df_file_bci['BCI'].values#[2:] 
            start_sec = offsets[file_base] #+ 20
            
            # Ensures that PPNet shape is conserved and matches point-to-point with BCI.
            # last two rows are omitted for 20s offset
            min_len = min(feat.shape[0], len(bci_aligned))
            if min_len == 0: continue
            
            feat = feat[:min_len]
            pred = pred[:min_len]
            acts = acts[:min_len]
            bci_aligned = bci_aligned[:min_len*5]

            # --- Calculate Cutoff & Remainder ---
            cutoff = (min_len // N) * N
            remainder = min_len % N

            # 1. Process the "Clean" Blocks (100s windows)
            if cutoff > 0:
                feat_c = feat[:cutoff].reshape(-1, N, feat.shape[1]).mean(axis=1)
                pred_c = stats.mode(pred[:cutoff].reshape(-1, N), axis=1, keepdims=False)[0]
                acts_c = acts[:cutoff].reshape(-1, N, acts.shape[1]).mean(axis=1)
                
                # Use nanmean to safely handle missing artifact data within the 100s window
                bci_c  = np.nanmean(bci_aligned[:cutoff].reshape(-1, N), axis=1)
                
                res_c  = np.full(feat_c.shape[0], 100.0) # Track resolution as float
                time_c = start_sec + (np.arange(0, cutoff, N) * 10)
            else:
                feat_c, pred_c, acts_c, bci_c, res_c, time_c = None, None, None, None, None, None

            # 2. Process the "Remainder" Block (No Data Loss)
            if remainder > 0:
                feat_r = feat[cutoff:].mean(axis=0, keepdims=True)
                pred_r = np.array([stats.mode(pred[cutoff:], keepdims=False)[0]])
                acts_r = acts[cutoff:].mean(axis=0, keepdims=True)
                
                # Use nanmean for the remainder block
                bci_r  = np.array([np.nanmean(bci_aligned[cutoff:])])
                
                res_r  = np.array([remainder * 10.0]) # Track actual resolution (e.g., 40.0)
                time_r = np.array([start_sec + cutoff * 10])

                # Combine clean blocks with remainder
                if feat_c is not None:
                    feat_c = np.vstack([feat_c, feat_r])
                    pred_c = np.concatenate([pred_c, pred_r])
                    acts_c = np.vstack([acts_c, acts_r])
                    bci_c  = np.concatenate([bci_c, bci_r])
                    res_c  = np.concatenate([res_c, res_r])
                    time_c = np.concatenate([time_c, time_r])
                else:
                    feat_c, pred_c, acts_c, bci_c, res_c, time_c = feat_r, pred_r, acts_r, bci_r, res_r, time_r

            # --- Apply BCI Sub-classification ---
            # 0: Other, maps to -> 6: Burst Suppression, 7: Continuous, 8: Discontinuous
            other_mask = (pred_c == 0)
            
            # Use ~np.isnan to ensure we don't accidentally classify a NaN BCI gap
            bs_mask   = other_mask & ~np.isnan(bci_c) & (bci_c < 0.5)
            disc_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.5) & (bci_c < 0.9)
            cont_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.9)

            pred_c[bs_mask] = 0
            pred_c[cont_mask] = 6
            pred_c[disc_mask] = 7

            # --- Metadata & Appending ---
            chunk_ids = time_c // SEC_PER_CHUNK
            pat_cpc = cpc_dict.get(patient_id, np.nan)

            all_features.append(feat_c)
            all_predictions.append(pred_c)
            all_activations.append(acts_c)
            all_time_chunks.append(chunk_ids)
            all_resolutions.append(res_c)
            all_times.append(time_c)
            all_patient_ids.extend([patient_id] * feat_c.shape[0])
            all_cpc.extend([pat_cpc] * feat_c.shape[0])

        except Exception as e:
            print(f"Error loading {file}: {e}")

# --- 3. Stack, Filter, and Chronologically Sort ---
# Stack everything
final_time_chunks = np.concatenate(all_time_chunks)
final_times       = np.concatenate(all_times)
final_features    = np.vstack(all_features)
final_predictions = np.concatenate(all_predictions)
final_activations = np.vstack(all_activations)
final_resolutions = np.concatenate(all_resolutions)
final_patient_ids = np.array(all_patient_ids)
final_cpc         = np.array(all_cpc)

# Create an index to sort chronologically per patient
df_sort = pd.DataFrame({'patient': final_patient_ids, 'time': final_times})
sort_idx = df_sort.sort_values(['patient', 'time']).index

# Apply sorting and the 84-hour (chunk < 14) filter simultaneously
valid_mask = final_time_chunks[sort_idx] < 14
final_idx = sort_idx[valid_mask]

# Final Arrays
out_features = final_features[final_idx]
out_predictions = final_predictions[final_idx]
out_activations = final_activations[final_idx]
out_time_chunks = final_time_chunks[final_idx]
out_times = final_times[final_idx]
out_resolutions = final_resolutions[final_idx]
out_patient_ids = final_patient_ids[final_idx]
out_cpc = final_cpc[final_idx]

# Save to NPZ
np.savez_compressed('4_20_2026_revised_ppnet_bci_cpc_time.npz', 
                    features=out_features, 
                    predictions=out_predictions, 
                    activations=out_activations,
                    chunks=out_time_chunks,
                    times=out_times,
                    resolutions=out_resolutions,
                    patient_ids=out_patient_ids,
                    cpc_scores=out_cpc)

print(f"Complete! Output shape: {out_predictions.shape}")
# Runtime: 45m

In [ ]:
#'----------------------
#' Script name: chunkify_ppnet.py 
#' 
#' Version: 0.0.3 (50s Resolution Update)
#' 
#' Authors: Michael P. Brown
#' 
#' Date created: 2026-04-05
#' 
#' Copyright (c) 2026
#' Email: mpbrown@berkeley.edu
#'----------------------
# --- 1. Configuration & Mappings ---
N = 10
SEC_PER_CHUNK = 21600 # 6 hours

# Directories
mgh_dir = "../ICARE_protopnet_results/MGH/"
ulb_dir = "../ICARE_protopnet_results/ULB/"
ynh_dir = "../ICARE_protopnet_results/YNH/"
bidmc_dir = "../ICARE_protopnet_results/BIDMC/"
bwh_dir = "../ICARE_protopnet_results/BWH/"
utw_dir = "../ICARE_protopnet_results/UTW/"
npz_directories = [mgh_dir, ulb_dir, ynh_dir, bidmc_dir, bwh_dir, utw_dir]

bci_csv_dir = "../1000_ICARE_patient_10s_94f_with_spike/" 

# Load Offsets
df_offset = pd.read_csv("../Vishnu_IIC_SparcNet_Analysis/physionet_files_mapped.csv")
df_offset['Extended_Ids'] = df_offset['ICARE_files'].apply(lambda r: str(r)[:-4])
offsets = dict(zip(df_offset["Extended_Ids"].values, df_offset["time_from_rosc"].values))

# Load CPC Scores
df_cpc = pd.read_csv('pats_cpc_outcomes.csv')
cpc_dict = dict(zip(df_cpc['pat_ICARE'], df_cpc['cpc']))

# Initialize Global Lists
all_features, all_predictions, all_activations = [], [], []
all_patient_ids, all_time_chunks, all_resolutions, all_cpc, all_times = [], [], [], [], []

# --- 2. Main Extraction Loop ---
for npz_dir in npz_directories:
    if not os.path.exists(npz_dir): continue
        
    for file in os.listdir(npz_dir):
        if not file.endswith(".npz"): continue
        file_base = file.replace(".npz", "")
        
        if file_base not in offsets: continue
            
        patient_id = "_".join(file_base.split('_')[:2]) # e.g., 'ICARE_0647'
        
        # Load Patient's BCI CSV
        bci_csv_path = os.path.join(bci_csv_dir, f"{patient_id}_rel10s_with_spike.csv")
        if not os.path.exists(bci_csv_path): continue
            
        try:
            # --- Load PPNet Data ---
            with np.load(os.path.join(npz_dir, file)) as data:
                feat = data["extracted_features"] if "extracted_features" in data else data["proto_features"]
                pred = np.atleast_1d(data["predictions"])
                acts = data['activations']
                
                if len(feat.shape) == 1:
                    feat, acts = feat.reshape(1, -1), acts.reshape(1, -1)
            
            # --- Load and Align BCI Data ---
            df_bci = pd.read_csv(bci_csv_path)
            # Filter rows specifically for this .mat recording
            df_file_bci = df_bci[df_bci['file'].str.contains(file_base, na=False)].sort_values('rel_sec')
            if df_file_bci.empty: continue
                
            bci_aligned = df_file_bci['BCI'].values 
            start_sec = offsets[file_base] # Removed +20 offset, start exactly at offset
            
            # 1 PPNet row (50s) = 5 BCI rows (10s)
            # Find the maximum number of perfectly matched 50s blocks
            bci_equivalent_rows = len(bci_aligned) // 5
            min_len = min(feat.shape[0], bci_equivalent_rows)
            
            if min_len == 0: continue
            
            # Truncate to exact matching boundaries
            feat = feat[:min_len]
            pred = pred[:min_len]
            acts = acts[:min_len]
            bci_aligned = bci_aligned[:min_len * 5] # Extract exactly 5x rows of BCI

            # --- Calculate Cutoff & Remainder ---
            cutoff = (min_len // N) * N
            remainder = min_len % N

            # 1. Process the "Clean" Blocks (500s windows = N*50s)
            if cutoff > 0:
                feat_c = feat[:cutoff].reshape(-1, N, feat.shape[1]).mean(axis=1)
                pred_c = stats.mode(pred[:cutoff].reshape(-1, N), axis=1, keepdims=False)[0]
                acts_c = acts[:cutoff].reshape(-1, N, acts.shape[1]).mean(axis=1)
                
                # BCI aggregates over 50 rows (N*5) to equal the 500s window
                bci_c  = np.nanmean(bci_aligned[:cutoff * 5].reshape(-1, N * 5), axis=1)
                
                res_c  = np.full(feat_c.shape[0], N * 50.0) # Track resolution as 500.0s
                time_c = start_sec + (np.arange(0, cutoff, N) * 50)
            else:
                feat_c, pred_c, acts_c, bci_c, res_c, time_c = None, None, None, None, None, None

            # 2. Process the "Remainder" Block (No Data Loss)
            if remainder > 0:
                feat_r = feat[cutoff:].mean(axis=0, keepdims=True)
                pred_r = np.array([stats.mode(pred[cutoff:], keepdims=False)[0]])
                acts_r = acts[cutoff:].mean(axis=0, keepdims=True)
                
                # Average the remaining tail of BCI data
                bci_r  = np.array([np.nanmean(bci_aligned[cutoff * 5:])])
                
                res_r  = np.array([remainder * 50.0]) # Track actual resolution (e.g., 200s)
                time_r = np.array([start_sec + cutoff * 50])

                # Combine clean blocks with remainder
                if feat_c is not None:
                    feat_c = np.vstack([feat_c, feat_r])
                    pred_c = np.concatenate([pred_c, pred_r])
                    acts_c = np.vstack([acts_c, acts_r])
                    bci_c  = np.concatenate([bci_c, bci_r])
                    res_c  = np.concatenate([res_c, res_r])
                    time_c = np.concatenate([time_c, time_r])
                else:
                    feat_c, pred_c, acts_c, bci_c, res_c, time_c = feat_r, pred_r, acts_r, bci_r, res_r, time_r

            # --- Apply BCI Sub-classification ---
            # Update 'Other' dynamically
            other_mask = (pred_c == 0) # Assumes base PPNet 'Other' is 0
            
            # Use ~np.isnan to ensure we don't accidentally classify a NaN BCI gap
            bs_mask   = other_mask & ~np.isnan(bci_c) & (bci_c < 0.5)
            disc_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.5) & (bci_c < 0.9)
            cont_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.9)

            # Updated labels per your instructions
            pred_c[bs_mask] = 0
            pred_c[cont_mask] = 6
            pred_c[disc_mask] = 7

            # --- Metadata & Appending ---
            chunk_ids = time_c // SEC_PER_CHUNK
            pat_cpc = cpc_dict.get(patient_id, np.nan)

            all_features.append(feat_c)
            all_predictions.append(pred_c)
            all_activations.append(acts_c)
            all_time_chunks.append(chunk_ids)
            all_resolutions.append(res_c)
            all_times.append(time_c)
            all_patient_ids.extend([patient_id] * feat_c.shape[0])
            all_cpc.extend([pat_cpc] * feat_c.shape[0])

        except Exception as e:
            print(f"Error loading {file}: {e}")

# --- 3. Stack, Filter, and Chronologically Sort ---
# Stack everything
final_time_chunks = np.concatenate(all_time_chunks)
final_times       = np.concatenate(all_times)
final_features    = np.vstack(all_features)
final_predictions = np.concatenate(all_predictions)
final_activations = np.vstack(all_activations)
final_resolutions = np.concatenate(all_resolutions)
final_patient_ids = np.array(all_patient_ids)
final_cpc         = np.array(all_cpc)

# Create an index to sort chronologically per patient
df_sort = pd.DataFrame({'patient': final_patient_ids, 'time': final_times})
sort_idx = df_sort.sort_values(['patient', 'time']).index

# Apply sorting and the 84-hour (chunk < 14) filter simultaneously
valid_mask = final_time_chunks[sort_idx] < 14
final_idx = sort_idx[valid_mask]

# Final Arrays
out_features = final_features[final_idx]
out_predictions = final_predictions[final_idx]
out_activations = final_activations[final_idx]
out_time_chunks = final_time_chunks[final_idx]
out_times = final_times[final_idx]
out_resolutions = final_resolutions[final_idx]
out_patient_ids = final_patient_ids[final_idx]
out_cpc = final_cpc[final_idx]

# Save to NPZ
np.savez_compressed('4_20_2026_revised_ppnet_bci_cpc_time.npz', 
                    features=out_features, 
                    predictions=out_predictions, 
                    activations=out_activations,
                    chunks=out_time_chunks,
                    times=out_times,
                    resolutions=out_resolutions,
                    patient_ids=out_patient_ids,
                    cpc_scores=out_cpc)

print(f"Complete! Output shape: {out_predictions.shape}")

In [ ]:
#'----------------------
#' Script name: chunkify_ppnet.py 
#' 
#' Version: 0.0.4 (300s Res + CEBRA Features)
#' 
#' Authors: Michael P. Brown
#' 
#' Date created: 2026-04-05
#' 
#' Copyright (c) 2026
#' Email: mpbrown@berkeley.edu
#'----------------------
import os
import pandas as pd
import numpy as np
from scipy import stats
import warnings

# Suppress expected RuntimeWarnings from taking nanmean of all-NaN slices
warnings.filterwarnings(action='ignore', message='Mean of empty slice')

# --- 1. Configuration & Mappings ---
N = 6
SEC_PER_CHUNK = 21600 # 6 hours

# --- NEW: CEBRA Features ---
cebra_cols = [
    'SignalSD', 'corrmean', 'meanskewamp', 'sdspectent', 'shanavg', 
    'thetaalphamean', 'BCI','SIQ', 'SIQ_delta', 'SIQ_beta', 
    'SIQ_alpha', 'SIQ_theta', 'lv_l5'
]

# Directories
mgh_dir = "../ICARE_protopnet_results/MGH/"
ulb_dir = "../ICARE_protopnet_results/ULB/"
ynh_dir = "../ICARE_protopnet_results/YNH/"
bidmc_dir = "../ICARE_protopnet_results/BIDMC/"
bwh_dir = "../ICARE_protopnet_results/BWH/"
utw_dir = "../ICARE_protopnet_results/UTW/"
npz_directories = [mgh_dir, ulb_dir, ynh_dir, bidmc_dir, bwh_dir, utw_dir]

bci_csv_dir = "../1000_ICARE_patient_10s_94f_with_spike/" 

# Load Offsets
df_offset = pd.read_csv("../Vishnu_IIC_SparcNet_Analysis/physionet_files_mapped.csv")
df_offset['Extended_Ids'] = df_offset['ICARE_files'].apply(lambda r: str(r)[:-4])
offsets = dict(zip(df_offset["Extended_Ids"].values, df_offset["time_from_rosc"].values))

# Load CPC Scores
df_cpc = pd.read_csv('ICARE_clinical_complete_cpc.csv')
cpc_dict = dict(zip(df_cpc['pat_ICARE'], df_cpc['cpc']))

# Initialize Global Lists
all_features, all_predictions, all_activations = [], [], []
all_patient_ids, all_time_chunks, all_resolutions, all_cpc, all_times = [], [], [], [], []
all_cebra = [] # --- NEW: Store CEBRA aggregates ---

# --- 2. Main Extraction Loop ---
for npz_dir in npz_directories:
    if not os.path.exists(npz_dir): continue
        
    for file in os.listdir(npz_dir):
        if not file.endswith(".npz"): continue
        file_base = file.replace(".npz", "")
        
        if file_base not in offsets: continue
            
        patient_id = "_".join(file_base.split('_')[:2]) # e.g., 'ICARE_0647'
        
        # Load Patient's BCI CSV
        bci_csv_path = os.path.join(bci_csv_dir, f"{patient_id}_rel10s_with_spike.csv")
        if not os.path.exists(bci_csv_path): continue
            
        try:
            # --- Load PPNet Data ---
            with np.load(os.path.join(npz_dir, file)) as data:
                feat = data["extracted_features"] if "extracted_features" in data else data["proto_features"]
                pred = np.atleast_1d(data["predictions"])
                acts = data['activations']
                
                if len(feat.shape) == 1:
                    feat, acts = feat.reshape(1, -1), acts.reshape(1, -1)
            
            # --- Load and Align BCI Data ---
            df_bci = pd.read_csv(bci_csv_path)
            # Filter rows specifically for this .mat recording
            df_file_bci = df_bci[df_bci['file'].str.contains(file_base, na=False)].sort_values('rel_sec')
            if df_file_bci.empty: continue
                
            bci_aligned = df_file_bci['BCI'].values 
            
            # --- NEW: Extract CEBRA columns ---
            # Using a list comprehension failsafe just in case a file is missing a column
            valid_cebra_cols = [c for c in cebra_cols if c in df_file_bci.columns]
            cebra_aligned = df_file_bci[valid_cebra_cols].values 
            
            start_sec = offsets[file_base] 
            
            # 1 PPNet row (50s) = 5 BCI rows (10s)
            bci_equivalent_rows = len(bci_aligned) // 5
            min_len = min(feat.shape[0], bci_equivalent_rows)
            
            if min_len == 0: continue
            
            # Truncate to exact matching boundaries
            feat = feat[:min_len]
            pred = pred[:min_len]
            acts = acts[:min_len]
            bci_aligned = bci_aligned[:min_len * 5] 
            cebra_aligned = cebra_aligned[:min_len * 5] # --- NEW: Truncate CEBRA data ---

            # --- Calculate Cutoff & Remainder ---
            cutoff = (min_len // N) * N
            remainder = min_len % N

            # 1. Process the "Clean" Blocks (300s windows = N*50s)
            if cutoff > 0:
                feat_c = feat[:cutoff].reshape(-1, N, feat.shape[1]).mean(axis=1)
                pred_c = stats.mode(pred[:cutoff].reshape(-1, N), axis=1, keepdims=False)[0]
                acts_c = acts[:cutoff].reshape(-1, N, acts.shape[1]).mean(axis=1)
                
                # BCI aggregates over 30 rows (N*5)
                bci_c  = np.nanmean(bci_aligned[:cutoff * 5].reshape(-1, N * 5), axis=1)
                
                # --- NEW: CEBRA aggregates ---
                # Reshape from (rows, features) to (chunks, N*5, features) then mean across the chunk
                num_features = cebra_aligned.shape[1]
                cebra_c = np.nanmean(cebra_aligned[:cutoff * 5].reshape(-1, N * 5, num_features), axis=1)
                
                res_c  = np.full(feat_c.shape[0], N * 50.0) 
                time_c = start_sec + (np.arange(0, cutoff, N) * 50)
            else:
                feat_c, pred_c, acts_c, bci_c, cebra_c, res_c, time_c = None, None, None, None, None, None, None

            # 2. Process the "Remainder" Block (No Data Loss)
            if remainder > 0:
                feat_r = feat[cutoff:].mean(axis=0, keepdims=True)
                pred_r = np.array([stats.mode(pred[cutoff:], keepdims=False)[0]])
                acts_r = acts[cutoff:].mean(axis=0, keepdims=True)
                
                # Average the remaining tail
                bci_r  = np.array([np.nanmean(bci_aligned[cutoff * 5:])])
                
                # --- NEW: CEBRA remainder ---
                # Mean across the remaining rows (axis 0), keeping dims so it stacks nicely
                cebra_r = np.nanmean(cebra_aligned[cutoff * 5:], axis=0, keepdims=True)
                
                res_r  = np.array([remainder * 50.0]) 
                time_r = np.array([start_sec + cutoff * 50])

                # Combine clean blocks with remainder
                if feat_c is not None:
                    feat_c = np.vstack([feat_c, feat_r])
                    pred_c = np.concatenate([pred_c, pred_r])
                    acts_c = np.vstack([acts_c, acts_r])
                    bci_c  = np.concatenate([bci_c, bci_r])
                    cebra_c = np.vstack([cebra_c, cebra_r]) # --- NEW: Stack CEBRA remainder ---
                    res_c  = np.concatenate([res_c, res_r])
                    time_c = np.concatenate([time_c, time_r])
                else:
                    feat_c, pred_c, acts_c, bci_c, cebra_c, res_c, time_c = feat_r, pred_r, acts_r, bci_r, cebra_r, res_r, time_r

            # --- Apply BCI Sub-classification ---
            other_mask = (pred_c == 0) 
            
            bs_mask   = other_mask & ~np.isnan(bci_c) & (bci_c < 0.5)
            disc_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.5) & (bci_c < 0.9)
            cont_mask = other_mask & ~np.isnan(bci_c) & (bci_c >= 0.9)

            pred_c[bs_mask] = 0
            pred_c[cont_mask] = 6
            pred_c[disc_mask] = 7

            # --- Metadata & Appending ---
            chunk_ids = time_c // SEC_PER_CHUNK
            pat_cpc = cpc_dict.get(patient_id, np.nan)

            all_features.append(feat_c)
            all_predictions.append(pred_c)
            all_activations.append(acts_c)
            all_cebra.append(cebra_c) # --- NEW: Append to global CEBRA list ---
            all_time_chunks.append(chunk_ids)
            all_resolutions.append(res_c)
            all_times.append(time_c)
            all_patient_ids.extend([patient_id] * feat_c.shape[0])
            all_cpc.extend([pat_cpc] * feat_c.shape[0])

        except Exception as e:
            print(f"Error loading {file}: {e}")

# --- 3. Stack, Filter, and Chronologically Sort ---
final_time_chunks = np.concatenate(all_time_chunks)
final_times       = np.concatenate(all_times)
final_features    = np.vstack(all_features)
final_predictions = np.concatenate(all_predictions)
final_activations = np.vstack(all_activations)
final_cebra       = np.vstack(all_cebra) # --- NEW: Stack all CEBRA blocks ---
final_resolutions = np.concatenate(all_resolutions)
final_patient_ids = np.array(all_patient_ids)
final_cpc         = np.array(all_cpc)

# Create an index to sort chronologically per patient
df_sort = pd.DataFrame({'patient': final_patient_ids, 'time': final_times})
sort_idx = df_sort.sort_values(['patient', 'time']).index

# Apply sorting and the 84-hour (chunk < 14) filter simultaneously
valid_mask = final_time_chunks[sort_idx] < 14
final_idx = sort_idx[valid_mask]

# Final Arrays
out_features = final_features[final_idx]
out_predictions = final_predictions[final_idx]
out_activations = final_activations[final_idx]
out_cebra = final_cebra[final_idx] # --- NEW: Final sorted/filtered array ---
out_time_chunks = final_time_chunks[final_idx]
out_times = final_times[final_idx]
out_resolutions = final_resolutions[final_idx]
out_patient_ids = final_patient_ids[final_idx]
out_cpc = final_cpc[final_idx]

# Save to NPZ
np.savez_compressed('4_24_2026_300s_ppnet_bci_cpc_time_cebra.npz', 
                    features=out_features, 
                    predictions=out_predictions, 
                    activations=out_activations,
                    cebra_features=out_cebra, # --- NEW: Saved under key 'cebra_features' ---
                    chunks=out_time_chunks,
                    times=out_times,
                    resolutions=out_resolutions,
                    patient_ids=out_patient_ids,
                    cpc_scores=out_cpc)

print(f"Output shape: {out_predictions.shape}")
print(f"CEBRA features shape: {out_cebra.shape}") # Should be (Num_Samples, 12)

In [ ]:
b2 =np.load("4_20_2026_revised_ppnet_bci_cpc_time.npz")
shapator(b2)

In [ ]:
patient_df = pd.DataFrame(np.unique(b2['patient_ids']))
uniq_pats = np.unique(b2['patient_ids'])

In [ ]:
dropna_mask = ~np.isin(np.unique(b2['patient_ids']), ['ICARE_0051', 'ICARE_0237', 'ICARE_0254', 'ICARE_0336','ICARE_0689', 'ICARE_0692', 'ICARE_0741', 'ICARE_0816','ICARE_0929', 'ICARE_0945'])

In [ ]:
clean_pats = uniq_pats[dropna_mask]
clean_pats

In [ ]:
mask=(b2['patient_ids'] == 'ICARE_0647')
np.unique(b2['cpc_scores'][mask], return_counts=True)


In [ ]:
train_ids= pd.read_csv('keaton_train_ids.csv')['patient_id'].values
test_ids= pd.read_csv('keaton_test_ids.csv')['patient_id'].values

In [ ]:
len(train_ids) /800

In [ ]:
def tt_split(b, date_:str, train:str, test:str):
    # b : data
    # date_ : in format: m_d_y
    # train : file name with *train* set patient ids
    # test : file name with *test* set patient ids
    # --------
    train_ids= pd.read_csv(train)['patient_id'].values
    test_ids= pd.read_csv(test)['patient_id'].values
    pat_ids = b['patient_ids']
    # splitting PPNet into train/test sets
    for name,tset in zip(['train', 'test'],[train_ids, test_ids]):
        mask_patient = np.isin(pat_ids, tset)
        filtered_data = {
            'features': b['features'][mask_patient],
            'predictions': b['predictions'][mask_patient],
            'activations': b['activations'][mask_patient],
            'chunks': b['chunks'][mask_patient],
            'times': b['times'][mask_patient],
            'resolutions':b['resolutions'][mask_patient],
            'patient_ids': b['patient_ids'][mask_patient],
            'cpc_scores': b['cpc_scores'][mask_patient]
        }
        np.savez_compressed(f"{date_}_{name}_revised_ppnet_bci_cpc_times.npz", **filtered_data)

In [ ]:
# 4_16_2026_train_ppnet_feats_preds_acts_bci_cpc_times_removedNaNCPCpats.npz

In [ ]:
b_train, b_test = np.load('4_16_2026_train_ppnet_feats_preds_acts_bci_cpc_times_removedNaNCPCpats.npz'),np.load('4_16_2026_test_ppnet_feats_preds_acts_bci_cpc_times.npz')

In [ ]:
shapator(b_train), shapator(b_test)

In [ ]:
mask=(b['patient_ids'] == 'ICARE_0647')
np.unique(b['cpc_scores'][mask], return_counts=True)

In [ ]:
b2['times'][mask]

In [ ]:
sfs = np.load('../ICARE_protopnet_results/BIDMC/ICARE_0647_20100520_142937.npz')
sfs2 = np.load('../ICARE_protopnet_results/BIDMC/ICARE_0647_20100525_235158.npz')
# "C:\Users\micha\OneDrive\Desktop\UCSF\ICARE_protopnet_results\BIDMC\ICARE_0647_20100525_235158.npz"

In [ ]:
# pat647 has 28 PPNet output files, max size 2033KB: 391 preds,
# allfeatures has ~49k rows, and 
# result post-preprocessing: 298138-40497=257,641s
len(sfs2['predictions'])

In [ ]:
mask=(np.isnan(b2['cpc_scores']))
np.unique(b2['patient_ids'][mask], return_counts=True)

In [ ]:
sum([315, 335, 344, 492, 557, 438, 440, 334, 466, 195])

In [ ]:
np.unique(b['predictions'])
# label_map = {1: 'Seizure', 2: 'LPD', 3: 'GPD', 4: 'LRDA', 5: 'GRDA', 6: 'Burst Suppression', 7: 'Continuous', 8: 'Discontinous'}

In [ ]:
shapator(b_train)

In [ ]:
def r_nan(data, filename:str):
    mask_nancpc = ~np.isnan(data['cpc_scores'])
    filterednancpc_data = {
        'features': data['features'][mask_nancpc],
        'predictions': data['predictions'][mask_nancpc],
        'activations': data['activations'][mask_nancpc],
        'chunks': data['chunks'][mask_nancpc],
        'times': data['times'][mask_nancpc],
        'resolutions':data['resolutions'][mask_nancpc],
        'patient_ids': data['patient_ids'][mask_nancpc],
        'cpc_scores': data['cpc_scores'][mask_nancpc]
    }
    np.savez_compressed(filename, **filterednancpc_data)

In [ ]:
r_nan(b2, "4_21_2026_500s_ppnet_bci_cpc_time.npz")

In [ ]:
j = np.load('4_21_2026_500s_ppnet_bci_cpc_time.npz')

In [ ]:
shapator(j)

In [ ]:
# checking if r_nan() removed nans correctly

In [ ]:
b_train['predictions'].shape[0] - sum(mask_nancpc)
# 4029 rows lost from missing CPC

In [ ]:
bm=(b['patient_ids']=='ICARE_0238')
np.count_nonzero(bm)

In [ ]:
for k in ['predictions','chunks','resolutions', 'cpc_scores']:
    val, counts = np.unique(b[k], return_counts=True)
    labels = [str(int(v)) if not np.isnan(v) else "NaN" for v in val]
    print(f"{k}'s unique values: {val}; counts: {counts}")
    plt.bar(labels, counts)
    plt.title(f"{k} distribution")
    plt.xlabel('Categories'), plt.ylabel('Counts')
    plt.show()
    
# # { 0: 'Burst Suppression', 
#   1: 'Seizure', 
#   2: 'LPD', 
#   3: 'GPD', 
#   4: 'LRDA', 
#   5: 'GRDA', 
#   6: 'Continuous', 
#   7: 'Discontinous'}

In [ ]:
fa= np.delete(final_activations, 31, axis=1)
print(f"{final_activations.shape} now {fa.shape} without Proto. 31")
# by removing proto31, remember to preserve prototype naming scheme 
# for all prototypes following i.e. now prototype 44 is 45 etc.
prototypelabels = list(range(45))
prototypelabels.remove(31)
fp = np.array(prototypelabels)

In [ ]:
filtered_data = {
    'features': final_features,
    'predictions': final_predictions,
    'patient_ids': final_patient_ids,
    'prototype_activations': fa,
    'chunks': final_time_chunks,
    'prototype_labels': fp
}
np.savez_compressed('4_6_2026_ppnet_chunkified.npz', **filtered_data)

In [ ]:
ch_d= np.load('4_6_2026_ppnet_chunkified.npz')
shapator(ch_d)
# resulted with fewer pts from truncating data past chunk 14

In [ ]:
# train_ids= pd.read_csv('keaton_train_ids.csv')['patient_id'].values
# # reducing full PPNet to match the train-set patient ids
# pat_ids = ch_d['patient_ids']
# mask_patient = np.isin(pat_ids, train_ids)
# filtered_data = {
#     'features': ch_d['features'][mask_patient],
#     'predictions': ch_d['predictions'][mask_patient],
#     'patient_ids': ch_d['patient_ids'][mask_patient],
#     'prototype_activations': ch_d['prototype_activations'][mask_patient],
#     'chunks': ch_d['chunks'][mask_patient],
#     'prototype_labels': ch_d['prototype_labels']
# }
# np.savez_compressed('4_11_2026_ppnet_chunks_protolabels_train.npz', **filtered_data)

In [ ]:
def plot_super_advanced_hypnogram(target_patient_id, patient_ids, preds, chunk_indices, hmm_probs=None, smoothing_window=1, total_chunks=14, chunk_hours=6):
    """
    Plots an aggregated block hypnogram based on absolute chunk indexing to respect temporal gaps.
    X-axis is scaled to absolute hours.
    """
    # Isolate patient data
    p_mask = (patient_ids == target_patient_id)
    p_preds = preds[p_mask]
    p_chunks = chunk_indices[p_mask]
    num_epochs = len(p_preds)
    
    if num_epochs == 0:
        print(f"No data found for patient {target_patient_id}")
        return
        
    # Apply Smoothing
    if smoothing_window > 1:
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    label_map = {1: 'Seizure', 2: 'LPD', 3: 'GPD', 4: 'LRDA', 5: 'GRDA', 6: 'Burst Suppression', 7: 'Continuous', 8: 'Discontinuous'}
    display_order = ['Burst Suppression', 'Discontinuous','Continuous', 'GRDA', 'LRDA', 'LPD', 'GPD', 'Seizure']
    y_map = {k: display_order.index(v) for k, v in label_map.items()}

    # Setup Subplots
    num_plots = 2 if hmm_probs is not None else 1
    fig, axes = plt.subplots(num_plots, 1, figsize=(15, 3.5 * num_plots), sharex=True)
    ax1 = axes[0] if num_plots == 2 else axes
    cmap = plt.get_cmap('tab10')

    # Absolute Chunk Gridlines mapped to hours
    xticks = np.arange(total_chunks + 1)
    xticklabels = [f"{i * chunk_hours}h" for i in xticks]

    # 1. Aggregate Block Plotting by Chunk
    for chunk_idx in np.unique(p_chunks):
        local_mask = (p_chunks == chunk_idx)
        chunk_preds = p_preds[local_mask]
        N_c = len(chunk_preds)
        
        start_fraction = 0.0
        for state, group in groupby(chunk_preds):
            length = sum(1 for _ in group)
            fraction = length / N_c  # Normalize width so each chunk takes exactly 1 unit of X space
            y_val = y_map.get(state, 0)
            
            # Map X position absolutely: chunk_idx + internal_fraction
            rect = patches.Rectangle((chunk_idx + start_fraction, y_val - 0.4), fraction, 0.8, 
                                     linewidth=1, edgecolor='none', facecolor=cmap(state), alpha=1.0)
            ax1.add_patch(rect)
            start_fraction += fraction

    # Format ax1
    ax1.set_xlim(0, total_chunks)
    ax1.set_ylim(-0.5, len(display_order) - 0.5)
    ax1.set_yticks(range(len(display_order)))
    ax1.set_yticklabels(display_order, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    ax1.set_title(f"Aggregate EEG State Progression: Patient {target_patient_id}", fontweight='bold')
    ax1.set_ylabel("Predicted State")

    for tick in xticks:
        ax1.axvline(x=tick, color='gray', linestyle=':', alpha=0.6,zorder=0)

    target_ax = ax1

    # Final X-Axis Formatting
    target_ax.set_xticks(xticks)
    target_ax.set_xticklabels(xticklabels)
    target_ax.set_xlabel("Time (Hours)")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_super_advanced_hypnogram('ICARE_0647', b['patient_ids'], b['predictions'], b['chunks'], smoothing_window=1)

In [ ]:
plot_super_advanced_hypnogram('ICARE_0687', b['patient_ids'], b['predictions'], b['chunks'])

In [ ]:
plot_super_advanced_hypnogram('ICARE_0001', b['patient_ids'], b['predictions'], b['chunks'])

In [ ]:
def plot_superduper_advanced_hypnogram(target_patient_id, patient_ids, preds, times, resolutions, hmm_probs=None, smoothing_window=1, total_hours=84):
    """
    Plots an aggregated block hypnogram based on absolute timestamps to respect ALL temporal gaps.
    """
    # Isolate patient data
    p_mask = (patient_ids == target_patient_id)
    p_preds = preds[p_mask]
    p_times = times[p_mask]
    p_resolutions = resolutions[p_mask]
    num_epochs = len(p_preds)
    
    if num_epochs == 0:
        print(f"No data found for patient {target_patient_id}")
        return
        
    # Apply Smoothing
    if smoothing_window > 1:
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # --- REVISED CLASSES ---
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 4: 'LRDA', 5: 'GRDA', 
        6: 'Continuous', 7: 'Discontinuous'
    } 
    display_order = ['Burst Suppression', 'Discontinuous', 'Continuous', 'GRDA', 'LRDA', 'LPD', 'GPD', 'Seizure']
    y_map = {k: display_order.index(v) for k, v in label_map.items()}

    # Setup Subplots
    num_plots = 2 if hmm_probs is not None else 1
    fig, axes = plt.subplots(num_plots, 1, figsize=(15, 3.5 * num_plots), sharex=True)
    ax1 = axes[0] if num_plots == 2 else axes
    cmap = plt.get_cmap('tab10')

    # Convert absolute seconds to absolute hours for the X-axis
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    # 1. Aggregate Block Plotting by Absolute Time
    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    for i in range(1, num_epochs):
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap (>1 second tolerance)
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] # Extend the block
        else:
            # Draw the accumulated block
            y_val = y_map.get(current_state, 0)
            rects.append(patches.Rectangle((start_h, y_val - 0.4), end_h - start_h, 0.8, 
                                           facecolor=cmap(current_state), edgecolor='none'))
            # Reset for next block
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    y_val = y_map.get(current_state, 0)
    rects.append(patches.Rectangle((start_h, y_val - 0.4), end_h - start_h, 0.8, facecolor=cmap(current_state), edgecolor='none'))
    
    for rect in rects:
        ax1.add_patch(rect)

    # Format ax1
    ax1.set_xlim(0, total_hours)
    ax1.set_ylim(-0.5, len(display_order) - 0.5)
    ax1.set_yticks(range(len(display_order)))
    ax1.set_yticklabels(display_order, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    ax1.set_title(f"Absolute EEG State Progression: Patient {target_patient_id}", fontweight='bold')
    ax1.set_ylabel("Predicted State")

    # Add 6-hour gridlines
    xticks = np.arange(0, total_hours + 1, 6)
    xticklabels = [f"{i}h" for i in xticks]
    for tick in xticks:
        ax1.axvline(x=tick, color='gray', linestyle=':', alpha=0.6, zorder=0)

    target_ax = ax1

    # Final X-Axis Formatting
    target_ax.set_xticks(xticks)
    target_ax.set_xticklabels(xticklabels)
    target_ax.set_xlabel("Time (Hours)")

    plt.tight_layout()
    plt.show()

In [ ]:
b=np.load('4_23_2026_300s_ppnet_bci_cpc_time.npz')

In [ ]:
plot_superduper_advanced_hypnogram('ICARE_0647', b['patient_ids'], b['predictions'], b['times'], b['resolutions'], smoothing_window=1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# ---------------------------------------------------------
# Publication-Ready Aesthetic Settings (Nature/Science style)
# ---------------------------------------------------------
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.linewidth': 1.2,
    'pdf.fonttype': 42, # TrueType fonts for easy illustrator editing
    'ps.fonttype': 42,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# Define the clinical severity order for axes
CLINICAL_ORDER = [6, 8, 7, 5, 4, 2, 3, 1]
LABEL_MAP = {
    6: 'Burst Suppression', 8: 'Discontinuous', 7: 'Continuous', 
    5: 'GRDA', 4: 'LRDA', 2: 'LPD', 3: 'GPD', 1: 'Seizure'
}
LABELS_ORDERED = [LABEL_MAP[i] for i in CLINICAL_ORDER]

In [ ]:
def plot_outcome_stratified_transitions(data):
    """Plots transition matrices split by Good (CPC 1-2) vs Poor (CPC 3-5) outcomes."""
    preds, times, res = data['predictions'], data['times'], data['resolutions']
    pats, cpc = data['patient_ids'], data['cpc_scores']
    
    def get_transitions(target_cpc_range):
        tm = np.zeros((9, 9)) # 1-8 indexing, 0 is unused buffer
        mask = np.isin(cpc, target_cpc_range)
        valid_pats = np.unique(pats[mask])
        
        for p in valid_pats:
            p_idx = (pats == p)
            p_pr, p_t, p_r = preds[p_idx], times[p_idx], res[p_idx]
            
            for i in range(len(p_pr) - 1):
                # Only count transition if there is no physical gap
                if (p_t[i+1] - (p_t[i] + p_r[i])) <= 1.0:
                    tm[p_pr[i], p_pr[i+1]] += 1
        return tm
    
    # Calculate and row-normalize
    tm_good = get_transitions([1, 2])
    tm_poor = get_transitions([3, 4, 5])
    
    # Reorder to clinical severity
    tm_good_ordered = tm_good[np.ix_(CLINICAL_ORDER, CLINICAL_ORDER)]
    tm_poor_ordered = tm_poor[np.ix_(CLINICAL_ORDER, CLINICAL_ORDER)]
    
    tm_good_norm = np.nan_to_num(tm_good_ordered / tm_good_ordered.sum(axis=1, keepdims=True))
    tm_poor_norm = np.nan_to_num(tm_poor_ordered / tm_poor_ordered.sum(axis=1, keepdims=True))
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    cmap = sns.color_palette("rocket_r", as_cmap=True)
    
    for ax, tm, title in zip(axes, [tm_good_norm, tm_poor_norm], ['Good Outcome (CPC 1-2)', 'Poor Outcome (CPC 3-5)']):
        sns.heatmap(tm, annot=True, fmt=".2f", cmap=cmap, xticklabels=LABELS_ORDERED, 
                    yticklabels=LABELS_ORDERED, vmin=0, vmax=1, ax=ax, cbar_kws={'shrink': 0.8})
        ax.set_title(title, fontweight='bold', pad=15)
        ax.set_xlabel("Transition To")
        ax.set_ylabel("Transition From")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        
    plt.tight_layout()
    plt.show()

In [ ]:
plot_outcome_stratified_transitions(b_train)

In [ ]:
from lifelines import KaplanMeierFitter

def plot_malignant_survival(data):
    """Kaplan-Meier curve for Time-to-First Malignant State (GPD or Seizure)."""
    preds, times, pats, cpc = data['predictions'], data['times'], data['patient_ids'], data['cpc_scores']
    
    unique_pats = np.unique(pats)
    events, durations, groups = [], [], []
    
    for p in unique_pats:
        idx = (pats == p)
        p_pr, p_t, p_cpc = preds[idx], times[idx], cpc[idx][0]
        
        if np.isnan(p_cpc): continue
            
        # Find first occurrence of GPD (3) or Seizure (1)
        malignant_mask = np.isin(p_pr, [1, 3])
        if np.any(malignant_mask):
            first_idx = np.argmax(malignant_mask)
            events.append(1) # Event occurred
            durations.append(p_t[first_idx] / 3600.0) # Absolute hour
        else:
            events.append(0) # Right-censored
            durations.append(p_t[-1] / 3600.0) # Last monitored hour
            
        groups.append('Good (CPC 1-2)' if p_cpc <= 2 else 'Poor (CPC 3-5)')
        
    df = pd.DataFrame({'duration': durations, 'event': events, 'group': groups})
    
    kmf = KaplanMeierFitter()
    fig, ax = plt.subplots(figsize=(7, 5))
    
    for group, color in zip(['Good (CPC 1-2)', 'Poor (CPC 3-5)'], ['#2ca02c', '#d62728']):
        mask = df['group'] == group
        kmf.fit(df['duration'][mask], df['event'][mask], label=group)
        kmf.plot_survival_function(ax=ax, color=color, linewidth=2, ci_alpha=0.15)
        
    ax.set_title("Time to First Malignant State (GPD/Seizure)", fontweight='bold')
    ax.set_xlabel("Time Post-ROSC (Hours)")
    ax.set_ylabel("Probability of Remaining Free of State")
    ax.set_xlim(0, 84)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_malignant_survival(b_train)

In [ ]:
def plot_pacmap_trajectories(data, pacmap_embeddings, target_pats):
    """
    Plots directed temporal trajectories for specific patients.
    pacmap_embeddings: (N, 2) array of pre-calculated embeddings matching data length.
    target_pats: list of patient strings e.g. ['ICARE_0647', 'ICARE_0297']
    """
    pats = data['patient_ids']
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot global background manifold (faded)
    ax.scatter(pacmap_embeddings[:, 0], pacmap_embeddings[:, 1], 
               s=5, color='lightgray', alpha=0.3, zorder=1, label='Global Manifold')
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for i, p in enumerate(target_pats):
        idx = np.where(pats == p)[0]
        if len(idx) == 0: continue
            
        p_emb = pacmap_embeddings[idx]
        cpc_val = data['cpc_scores'][idx[0]]
        
        # Plot points
        ax.scatter(p_emb[:, 0], p_emb[:, 1], s=30, color=colors[i], 
                   zorder=3, edgecolor='white', label=f"{p} (CPC {cpc_val})")
        
        # Draw directed trajectory arrows
        for j in range(len(p_emb) - 1):
            ax.annotate('', xy=p_emb[j+1], xytext=p_emb[j],
                        arrowprops=dict(arrowstyle="->", color=colors[i], lw=1.5, alpha=0.7),
                        zorder=2)
            
    ax.set_title("Temporal Electrophysiological Trajectories", fontweight='bold')
    ax.set_xlabel("PaCMAP Dimension 1")
    ax.set_ylabel("PaCMAP Dimension 2")
    ax.legend(frameon=False)
    ax.set_xticks([])
    ax.set_yticks([]) # Hide axes ticks for pure spatial representation
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()

In [ ]:
embd = np.load('revise_train_pacmap_embedding.npy')

In [ ]:
plot_pacmap_trajectories(b_train, embd, ['ICARE_0647'])

In [ ]:
def plot_activation_evolution(data, target_prototype_idx=0):
    """Plots the rolling activation magnitude of a specific prototype over absolute time."""
    acts, times, cpc = data['activations'], data['times'], data['cpc_scores']
    
    # Extract absolute hours and target prototype magnitude
    t_hours = times / 3600.0
    p_acts = acts[:, target_prototype_idx]
    
    df = pd.DataFrame({'Hour': t_hours, 'Activation': p_acts, 'CPC': cpc})
    df = df.dropna(subset=['CPC'])
    df['Outcome'] = np.where(df['CPC'] <= 2, 'Good (CPC 1-2)', 'Poor (CPC 3-5)')
    
    # Bin by 6-hour windows for smoothing
    df['Time_Bin'] = (df['Hour'] // 6) * 6
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=df, x='Time_Bin', y='Activation', hue='Outcome', 
                 palette={'Good (CPC 1-2)': '#2ca02c', 'Poor (CPC 3-5)': '#d62728'}, 
                 marker='o', lw=2, ax=ax, errorbar=('ci', 95))
    ax.set_title(f"Evolution of Prototype {target_prototype_idx} Activation", fontweight='bold')
    ax.set_xlabel("Time Post-ROSC (Hours)")
    ax.set_ylabel("Mean Activation Magnitude")
    ax.set_xlim(0, 84)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.legend(frameon=False, title="Outcome")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_activation_evolution(b_train,15), plot_activation_evolution(b_train,29)

In [ ]:
def plot_activation_evolution_by_class(data, target_prototype_idx=0):
    """Plots the rolling activation magnitude of a specific prototype over absolute time, grouped by EEG State."""
    acts, times, preds = data['activations'], data['times'], data['predictions']
    
    # Extract absolute hours and target prototype magnitude
    t_hours = times / 3600.0
    p_acts = acts[:, target_prototype_idx]
    
    # Build DataFrame
    df = pd.DataFrame({'Hour': t_hours, 'Activation': p_acts, 'Prediction': preds})
    
    # Map the integers to your updated clinical labels
    label_map = {
        0: 'Burst Suppression', 
        1: 'Seizure', 
        2: 'LPD', 
        3: 'GPD', 
        4: 'LRDA', 
        5: 'GRDA', 
        6: 'Continuous', 
        7: 'Discontinuous'
    }
    df['State'] = df['Prediction'].map(LABEL_MAP)
    df = df.dropna(subset=['State']) # Drop any unexpected/unmapped values
    
    # Bin by 6-hour windows for smoothing
    df['Time_Bin'] = (df['Hour'] // 6) * 6
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot with a multi-class color palette
    sns.lineplot(data=df, x='Time_Bin', y='Activation', hue='State', 
                 palette='tab10', marker='o', lw=2, ax=ax, errorbar=('ci', 95))
    
    ax.set_title(f"Evolution of Prototype {target_prototype_idx} Activation by Predicted State", fontweight='bold')
    ax.set_xlabel("Time Post-ROSC (Hours)")
    ax.set_ylabel("Mean Activation Magnitude")
    ax.set_xlim(0, 84)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Move legend outside the plot to prevent overlapping with the 8 lines
    ax.legend(frameon=False, title="Predicted State", loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_activation_evolution_by_class(b_train, 29)

In [ ]:
def plot_missingness_biomarker(data):
    """Boxplot comparing the total hours of EEG recorded vs CPC Outcome."""
    pats, res, cpc = data['patient_ids'], data['resolutions'], data['cpc_scores']
    
    unique_pats = np.unique(pats)
    total_hours, outcomes = [], []
    
    for p in unique_pats:
        idx = (pats == p)
        p_cpc = cpc[idx][0]
        if np.isnan(p_cpc): continue
            
        # Sum total recorded seconds and convert to hours
        t_hrs = np.sum(res[idx]) / 3600.0
        
        total_hours.append(t_hrs)
        outcomes.append(f"CPC {int(p_cpc)}")
        
    df = pd.DataFrame({'Total_Recorded_Hours': total_hours, 'CPC': outcomes})
    df = df.sort_values('CPC')
    
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=df, x='CPC', y='Total_Recorded_Hours', 
                palette="mako", width=0.5, fliersize=3, ax=ax)
    sns.stripplot(data=df, x='CPC', y='Total_Recorded_Hours', 
                  color=".25", alpha=0.4, size=4, ax=ax)
    
    ax.set_title("Recording Density by Clinical Outcome", fontweight='bold')
    ax.set_xlabel("Discharge CPC Score")
    ax.set_ylabel("Total EEG Recorded (Hours)")
    ax.set_ylim(0, 84) # Max possible is 84
    plt.tight_layout()
    plt.show()

In [ ]:
plot_missingness_biomarker(b_train)

In [ ]:
pdat = np.load('3_15_2026_ppnet_pacmap_data.npz')
shapator(pdat)

In [ ]:
def plot_single_band_hypnogram(target_patient_id, data, smoothing_window=1, total_hours=84):
    """
    Plots an absolute timeline hypnogram as a single continuous color-coded band.
    """
    # Isolate patient data
    pats = data['patient_ids']
    p_mask = (pats == target_patient_id)
    
    if not np.any(p_mask):
        print(f"No data found for patient {target_patient_id}")
        return
        
    p_preds = data['predictions'][p_mask]
    p_times = data['times'][p_mask]
    p_resolutions = data['resolutions'][p_mask]
    num_epochs = len(p_preds)
    
    # Apply Smoothing (if requested)
    if smoothing_window > 1:
        from scipy.signal import medfilt
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # Label Map (Updated for 0-7 classes)
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 
        4: 'LRDA', 5: 'GRDA', 6: 'Continuous', 7: 'Discontinuous'
    }
    
    # Use tab10 for distinct, publication-safe categorical colors
    cmap = plt.get_cmap('tab10')
    
    # Setup Figure - Shorter height for a single band
    fig, ax = plt.subplots(figsize=(14, 2.5)) 
    
    # Convert absolute seconds to absolute hours
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    # Track which states actually appear for a clean, dynamic legend
    states_present = set([current_state])
    
    # 1. Aggregate Block Plotting by Absolute Time
    for i in range(1, num_epochs):
        states_present.add(p_preds[i])
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap, extend the block
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] 
        else:
            # Draw the accumulated block at y=0 with height=1
            rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                           facecolor=cmap(current_state), edgecolor='none'))
            # Reset for next block
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                   facecolor=cmap(current_state), edgecolor='none'))
    
    for rect in rects:
        ax.add_patch(rect)

    # Format Axes
    ax.set_xlim(0, total_hours)
    ax.set_ylim(0, 1) # Lock to the height of the single band
    
    # Hide Y-axis ticks/labels, just leave the axis title
    ax.set_yticks([]) 
    ax.set_ylabel("Predicted State", fontweight='bold', labelpad=15)

    # Format X-Axis (6-hour intervals)
    xticks = np.arange(0, total_hours + 1, 6)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{i}h" for i in xticks])
    ax.set_xlabel("Time Post-ROSC (Hours)")
    ax.set_title(f"Absolute EEG Timeline: Patient {target_patient_id}", fontweight='bold', pad=15)

    # Add vertical gridlines for easy time tracking
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle=':', alpha=0.5, zorder=0)

    # Create Dynamic Legend
    legend_patches = [
        patches.Patch(color=cmap(state), label=label_map.get(state, f"State {state}"))
        for state in sorted(list(states_present))
    ]
    
    # Anchor the legend below the plot
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.9, -0.4),
              ncol=min(len(states_present), 4), frameon=False)

    # Clean up spines (borders)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def psb(target_patient_id, data, smoothing_window=1, total_hours=84):
    """
    Plots an absolute timeline hypnogram as a thin, single continuous color-coded band.
    """
    # Isolate patient data
    pats = data['patient_ids']
    p_mask = (pats == target_patient_id)
    
    if not np.any(p_mask):
        print(f"No data found for patient {target_patient_id}")
        return
        
    p_preds = data['predictions'][p_mask]
    p_times = data['times'][p_mask]
    p_resolutions = data['resolutions'][p_mask]
    num_epochs = len(p_preds)
    
    # Apply Smoothing (if requested)
    if smoothing_window > 1:
        from scipy.signal import medfilt
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # Label Map
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 
        4: 'LRDA', 5: 'GRDA', 6: 'Continuous', 7: 'Discontinuous'
    }
    
    # PPNet Specific Color Mapping
    ppnet_colors = {
        0: '#7C6494', # Burst-Suppression
        1: '#FF4040', # Seizure
        2: '#35D2BA', # LPD
        3: '#449C7C', # GPD
        4: '#F9EBB2', # LRDA
        5: '#F8D759', # GRDA
        6: '#B3CDF5', # Continuous
        7: '#C3B2EC'  # Discontinuous
    }
    
    # Uniformly increase font size
    plt.rcParams.update({'font.size': 12})
    
    # Setup Figure 
    fig, ax = plt.subplots(figsize=(15, 3.5)) 
    
    # Convert absolute seconds to absolute hours
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    # Track which states actually appear for a clean, dynamic legend
    states_present = set([current_state])
    
    # Aggregate Block Plotting by Absolute Time
    for i in range(1, num_epochs):
        states_present.add(p_preds[i])
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap, extend the block
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] 
        else:
            rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                           facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                   facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
    
    for rect in rects:
        ax.add_patch(rect)

    # Format Axes
    ax.set_xlim(0, total_hours)
    ax.set_ylim(0, 1)
    
    ax.set_yticks([]) 
    ax.set_ylabel("Predicted State", fontweight='bold', labelpad=15)

    # Format X-Axis (6-hour intervals)
    xticks = np.arange(0, total_hours + 1, 6)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{i}h" for i in xticks])
    ax.set_xlabel("Time Post-ROSC (Hours)", fontweight='bold', labelpad=10)
    
    # Title
    ax.set_title(f"Absolute EEG Timeline: Patient {target_patient_id}", fontweight='bold', pad=25)

    # Add vertical gridlines for easy time tracking
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle=':', alpha=0.5, zorder=0)

    # Create Dynamic Legend
    legend_patches = [
        patches.Patch(color=ppnet_colors.get(state), label=label_map.get(state, f"State {state}"))
        for state in sorted(list(states_present))
    ]
    
    # Pushed the legend much further down by changing y from -0.6 to -1.2
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -1.2),
              ncol=min(len(states_present), 4), frameon=False, columnspacing=1.5)

    # Clean up spines (borders)
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    # Adjusted margins: slightly more room at the bottom so the legend doesn't get clipped
    plt.subplots_adjust(left=0.05, right=0.95, top=0.6, bottom=0.45)
    
    plt.show()

In [ ]:
b=np.load('4_27_2026_300s_ppnet_bci_cpc_time_cebra_logits_probs.npz')

In [ ]:
plot_single_band_hypnogram('ICARE_0647', b, smoothing_window=12), plot_superduper_advanced_hypnogram('ICARE_0647', b['patient_ids'], b['predictions'], b['times'], b['resolutions'], smoothing_window=12)

In [ ]:
psb('ICARE_0647', b, smoothing_window=12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def psb1(target_patient_id, data, smoothing_window=1, total_hours=84):
    """
    Plots an absolute timeline hypnogram as a thin, single continuous color-coded band.
    """
    # Isolate patient data
    pats = data['patient_ids']
    p_mask = (pats == target_patient_id)
    
    if not np.any(p_mask):
        print(f"No data found for patient {target_patient_id}")
        return
        
    p_preds = data['predictions'][p_mask]
    p_times = data['times'][p_mask]
    p_resolutions = data['resolutions'][p_mask]
    num_epochs = len(p_preds)
    
    # Apply Smoothing (if requested)
    if smoothing_window > 1:
        from scipy.signal import medfilt
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # Label Map
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 
        4: 'LRDA', 5: 'GRDA', 6: 'Continuous', 7: 'Discontinuous'
    }
    
    # PPNet Specific Color Mapping
    ppnet_colors = {
        0: '#7C6494', 
        1: '#FF4040', 
        2: '#35D2BA', 
        3: '#449C7C', 
        4: '#F9EBB2', 
        5: '#F8D759', 
        6: '#B3CDF5', 
        7: '#C3B2EC'  
    }
    
    # Uniformly increase font size
    plt.rcParams.update({'font.size': 12})
    
    # Setup Figure: Reduced height from 2.5 to 1.2 to make the band strictly thinner
    fig, ax = plt.subplots(figsize=(15, 1.2)) 
    
    # Convert absolute seconds to absolute hours
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    # Aggregate Block Plotting by Absolute Time
    for i in range(1, num_epochs):
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap, extend the block
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] 
        else:
            rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                           facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                   facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
    
    for rect in rects:
        ax.add_patch(rect)

    # Format Axes
    ax.set_xlim(0, total_hours)
    ax.set_ylim(0, 1)
    ax.set_yticks([]) 
    
    # Labels and Title
    ax.set_ylabel("Predicted State", fontweight='bold', labelpad=15)
    ax.set_xlabel("Time Post-ROSC (Hours)", fontweight='bold', labelpad=10)
    ax.set_title(f"Absolute EEG Timeline: Patient {target_patient_id}", fontweight='bold', pad=20)

    # Force strict 6-hour interval formatting on the X-axis
    xticks = np.arange(0, total_hours + 1, 6)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{i}h" for i in xticks])

    # Add vertical gridlines
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle=':', alpha=0.5, zorder=0)

    # Create Static Legend with all classes
    legend_patches = [
        patches.Patch(color=ppnet_colors[state], label=label_map[state])
        for state in sorted(label_map.keys())
    ]
    
    # Anchor legend. Adjusted y-offset (-1.1) to account for the shorter figure height
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -1.1),
              ncol=4, frameon=False, columnspacing=1.5)

    # Clean up spines
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
psb1('ICARE_0647', b, smoothing_window=12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def psb2(target_patient_id, data, smoothing_window=1, total_hours=84):
    """
    Plots an absolute timeline hypnogram as a thin, single continuous color-coded band.
    """
    # Isolate patient data
    pats = data['patient_ids']
    p_mask = (pats == target_patient_id)
    
    if not np.any(p_mask):
        print(f"No data found for patient {target_patient_id}")
        return
        
    p_preds = data['predictions'][p_mask]
    p_times = data['times'][p_mask]
    p_resolutions = data['resolutions'][p_mask]
    num_epochs = len(p_preds)
    
    # Apply Smoothing (if requested)
    if smoothing_window > 1:
        from scipy.signal import medfilt
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # Label Map
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 
        4: 'LRDA', 5: 'GRDA', 6: 'Continuous', 7: 'Discontinuous'
    }
    
    # PPNet Specific Color Mapping
    ppnet_colors = {
        0: '#7C6494', 
        1: '#FF4040', 
        2: '#35D2BA', 
        3: '#449C7C', 
        4: '#F9EBB2', 
        5: '#F8D759', 
        6: '#B3CDF5', 
        7: '#C3B2EC'  
    }
    
    # Uniformly increase font size
    plt.rcParams.update({'font.size': 12})
    
    # Setup Figure: Reduced height to 0.8 for a slightly thinner bar
    fig, ax = plt.subplots(figsize=(15, 0.8)) 
    
    # Convert absolute seconds to absolute hours
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    # Aggregate Block Plotting by Absolute Time
    for i in range(1, num_epochs):
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap, extend the block
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] 
        else:
            rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                           facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                   facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
    
    for rect in rects:
        ax.add_patch(rect)

    # Format Axes
    ax.set_xlim(0, total_hours)
    ax.set_ylim(0, 1)
    ax.set_yticks([]) 
    
    # Labels and Title
    ax.set_ylabel("Predicted State", fontweight='bold', labelpad=15)
    ax.set_xlabel("Time Post-ROSC (Hours)", fontweight='bold', labelpad=10)
    ax.set_title(f"Absolute EEG Timeline: Patient {target_patient_id}", fontweight='bold', pad=20)

    # Force strict 6-hour interval formatting on the X-axis
    xticks = np.arange(0, total_hours + 1, 6)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{i}h" for i in xticks])

    # Add vertical gridlines
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle=':', alpha=0.5, zorder=0)

    # Create Static Legend with all classes
    legend_patches = [
        patches.Patch(color=ppnet_colors[state], label=label_map[state])
        for state in sorted(label_map.keys())
    ]
    
    # Anchor legend: Adjusted y-offset closer to 0 to reduce the gap
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -0.95),
              ncol=8, frameon=False, columnspacing=1.5)

    # Clean up spines
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
psb2('ICARE_0647', b, smoothing_window=12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def psb21(target_patient_id, data, smoothing_window=1, total_hours=84):
    """
    Plots an absolute timeline hypnogram as a thin, single continuous color-coded band.
    """
    # Isolate patient data
    pats = data['patient_ids']
    p_mask = (pats == target_patient_id)
    
    if not np.any(p_mask):
        print(f"No data found for patient {target_patient_id}")
        return
        
    p_preds = data['predictions'][p_mask]
    p_times = data['times'][p_mask]
    p_resolutions = data['resolutions'][p_mask]
    num_epochs = len(p_preds)
    
    # Apply Smoothing (if requested)
    if smoothing_window > 1:
        from scipy.signal import medfilt
        kernel = smoothing_window if smoothing_window % 2 != 0 else smoothing_window + 1
        p_preds = medfilt(p_preds, kernel_size=kernel).astype(int)

    # Label Map
    label_map = {
        0: 'Burst Suppression', 1: 'Seizure', 2: 'LPD', 3: 'GPD', 
        4: 'LRDA', 5: 'GRDA', 6: 'Continuous', 7: 'Discontinuous'
    }
    
    # PPNet Specific Color Mapping
    ppnet_colors = {
        0: '#7C6494', 
        1: '#FF4040', 
        2: '#35D2BA', 
        3: '#449C7C', 
        4: '#F9EBB2', 
        5: '#F8D759', 
        6: '#B3CDF5', 
        7: '#C3B2EC'  
    }
    
    # Uniformly increase font size
    plt.rcParams.update({'font.size': 12})
    
    # Setup Figure: Reduced height to 0.8 for a slightly thinner bar
    fig, ax = plt.subplots(figsize=(15, 0.8)) 
    
    # Convert absolute seconds to absolute hours
    t_hours = p_times / 3600.0
    r_hours = p_resolutions / 3600.0

    rects = []
    current_state = p_preds[0]
    start_h = t_hours[0]
    end_h = t_hours[0] + r_hours[0]
    
    # Aggregate Block Plotting by Absolute Time
    for i in range(1, num_epochs):
        time_gap = t_hours[i] - end_h
        
        # If the state is the same AND there is no temporal gap, extend the block
        if p_preds[i] == current_state and time_gap < (1.0 / 3600.0):
            end_h = t_hours[i] + r_hours[i] 
        else:
            rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                           facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
            current_state = p_preds[i]
            start_h = t_hours[i]
            end_h = t_hours[i] + r_hours[i]
            
    # Draw the final block
    rects.append(patches.Rectangle((start_h, 0), end_h - start_h, 1, 
                                   facecolor=ppnet_colors.get(current_state, '#000000'), edgecolor='none'))
    
    for rect in rects:
        ax.add_patch(rect)

    # Format Axes
    ax.set_xlim(0, total_hours)
    ax.set_ylim(0, 1)
    ax.set_yticks([]) 
    
    # Labels and Title
    ax.set_ylabel("Predicted State", fontweight='bold', labelpad=15)
    ax.set_xlabel("Time Post-ROSC (Hours)", fontweight='bold', labelpad=10)
    ax.set_title(f"Absolute EEG Timeline: Patient {target_patient_id}", fontweight='bold', pad=20)

    # Force strict 6-hour interval formatting on the X-axis
    xticks = np.arange(0, total_hours + 1, 6)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{i}h" for i in xticks])

    # Add vertical gridlines
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle=':', alpha=0.5, zorder=0)

    # --- THE FIX: Custom Legend Order ---
    # Seizure(1) > GPD(3) > LPD(2) > LRDA(4) > GRDA(5) > Burst Suppression(0) > Discontinuous(7) > Continuous(6)
    custom_order = [1, 3, 2, 4, 5, 0, 7, 6]
    
    legend_patches = [
        patches.Patch(color=ppnet_colors[state], label=label_map[state])
        for state in custom_order
    ]
    
    # Anchor legend: Adjusted y-offset closer to 0 to reduce the gap
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -0.95),
              ncol=8, frameon=False, columnspacing=1.5)

    # Clean up spines
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
psb21('ICARE_0647', b, smoothing_window=12)